In [1]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

In [2]:
# Environment & Paths
load_dotenv()

BASE_DIR      = Path().resolve().parent
RAW_DATA_DIR  = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_FILE_NAME = os.getenv("RAW_FILE_NAME")
RAW_DATA      = RAW_DATA_DIR / RAW_FILE_NAME

In [3]:
# Load Data and view first 5 rows
df = pd.read_csv(RAW_DATA)
print(df.head())
print(df.shape)

   client_id  age        job  marital    education credit_default mortgage  \
0          0   56  housemaid  married     basic.4y             no       no   
1          1   57   services  married  high.school        unknown       no   
2          2   37   services  married  high.school             no      yes   
3          3   40     admin.  married     basic.6y             no       no   
4          4   56   services  married  high.school             no       no   

  month  day  contact_duration  number_contacts  previous_campaign_contacts  \
0   may   13               261                1                           0   
1   may   19               149                1                           0   
2   may   23               226                1                           0   
3   may   27               151                1                           0   
4   may    3               307                1                           0   

  previous_outcome  cons_price_idx  euribor_three_months

In [4]:
## Create Subsets
# clients
target_cols_clients = ["client_id", "age", "job", "marital", "education", "credit_default","mortgage"]
df_clients = df[target_cols_clients]

# campaigns
target_cols_campaigns = ["client_id", "number_contacts", "contact_duration", "previous_campaign_contacts", "previous_outcome","campaign_outcome"]
df_campaigns = df[target_cols_campaigns]

# economics
target_cols_economics = ["client_id", "cons_price_idx","euribor_three_months"]
df_economics = df[target_cols_economics]

In [5]:
## Clean and transform data for clients
# For "job" column: change "." to "_"
df_clients["job"] = df_clients["job"].str.replace(".","_", regex=False)

# For "education" column: change "." to "_" and "unknown" to np.NaN
df_clients["education"] = (df_clients["education"].str.replace(".", "_", regex=False).replace("unknown", np.nan))

# Convert yes/no columns to boolean
bool_columns = ["credit_default", "mortgage"]

for col in bool_columns:
    df_clients[col] = (df_clients[col]
        .map({"yes": True, "no": False, "unknown": False})
        .astype(bool)
    )

print(df_clients.head())

   client_id  age        job  marital    education  credit_default  mortgage
0          0   56  housemaid  married     basic_4y           False     False
1          1   57   services  married  high_school           False     False
2          2   37   services  married  high_school           False      True
3          3   40     admin_  married     basic_6y           False     False
4          4   56   services  married  high_school           False     False


In [6]:
## Clean and transform data for campaigns
# Create a new colum for "year" in the main data frame
df["year"] = 2022

# Create a new column called "last_contact_date" using the columns "day" and "month" and "year" for the campaign data frame
df_campaigns["last_contact_date"] = pd.to_datetime(df["year"].astype(str) + "-" + df["month"] + "-" + df["day"].astype(str), format="%Y-%b-%d")

# Convert outcome columns to boolean
outcome_map = {
    "campaign_outcome":  {"yes": True, "no": False},
    "previous_outcome":  {"success": True, "failure": False, "nonexistent": False}
}

for col, mapping in outcome_map.items():
    df_campaigns[col] = df_campaigns[col].map(mapping).astype(bool)

print(df_campaigns.head())

   client_id  number_contacts  contact_duration  previous_campaign_contacts  \
0          0                1               261                           0   
1          1                1               149                           0   
2          2                1               226                           0   
3          3                1               151                           0   
4          4                1               307                           0   

   previous_outcome  campaign_outcome last_contact_date  
0             False             False        2022-05-13  
1             False             False        2022-05-19  
2             False             False        2022-05-23  
3             False             False        2022-05-27  
4             False             False        2022-05-03  


In [7]:
# Print value counts for target columns
for col in ["credit_default", "mortgage", "previous_outcome", "campaign_outcome"]:
    print(col)
    print("--------------")
    print(df[col].value_counts())

credit_default
--------------
credit_default
no         32588
unknown     8597
yes            3
Name: count, dtype: int64
mortgage
--------------
mortgage
yes        21576
no         18622
unknown      990
Name: count, dtype: int64
previous_outcome
--------------
previous_outcome
nonexistent    35563
failure         4252
success         1373
Name: count, dtype: int64
campaign_outcome
--------------
campaign_outcome
no     36548
yes     4640
Name: count, dtype: int64


In [8]:
# Export Processed Data
df_clients.to_csv(PROCESSED_DIR / "client.csv", index=False)
df_campaigns.to_csv(PROCESSED_DIR / "campaign.csv", index=False)
df_economics.to_csv(PROCESSED_DIR / "economics.csv", index=False)

print("\nFiles saved to data/processed/")


Files saved to data/processed/
